<div style="text-align: justify; max-width: 90ch">

# Lecture 01 exercises

Exercises are tagged with the goal they practise (see `reading_material/goals_01.md`); `[O]` marks stretch exercises. Time bands:
15 min (quick), 45 min (realistic), 90 min (stretch). Run the configuration cell first; Ollama must be running with the models pulled, as in the lectures.

</div>

In [ ]:
import json
import subprocess

import ollama
import pandas as pd
from huggingface_hub import HfApi, ModelCard
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

In [ ]:
# --- Course configuration: edit TIER only; settings in .env (01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

import os
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # your settings file: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-exercises"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

<div style="text-align: justify; max-width: 90ch">

### Exercise 1.1 [G1, G7] Your hardware report (15 min)

Run `llmfit` from Python (as in `lec_01b` section 5) and show the five best-fitting models for your machine. In a Markdown cell below the table,
state which `TIER` you will use in this course and give two reasons taken from the table (memory, speed, or fit).

</div>

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Your tier and reasons here.*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 1.2 [G5, G6, G7] Choose a model for a business scenario (45 min)

Pick **one** scenario:

- **A. Greek customer support.** A retailer wants an assistant that answers customer questions in Greek, on a server inside the company (privacy),
  on a machine with 16 GB of RAM and no GPU.
- **B. Code assistant for a product.** A startup will ship a coding helper inside its commercial product; English only; it must be allowed to sell it;
  latency matters more than size.
- **C. Contract summaries.** A law office wants summaries of 50-page contracts (English); nothing may leave the office; a workstation with a 24 GB GPU is available.

1. Write the requirements table for the scenario in a Markdown cell (the eight rows from `lec_01b` section 1).
2. Search the Hub with filters that match at least two requirements (language tag, `pipeline_tag`, a size or format keyword) and show the top results.
3. Load the model cards of two candidates and show license, languages and base model.
4. For your preferred candidate, list its GGUF quantizations and sizes, and pick one that fits the scenario's hardware.
5. Write the five-line decision record.

</div>

<div style="text-align: justify; max-width: 90ch">

*Requirements table here.*

</div>

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

*Decision record here.*

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 1.3 [G2, G5] Compare two models on five fixed questions (45 min)

Reuse the `compare` function from `lec_01b` section 6 (copy it here). Compare `qwen3:1.7b` with a second model you have pulled (`qwen3:0.6b`,
or the candidate from Exercise 1.2) on five questions of your own: three about the Python course, one in Greek, one that needs a short calculation.
Show the table and write two sentences: which requirement each model fails, if any.

</div>

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

### Exercise 1.4 [G4] Temperature sweep (15 min)

For the prompt "Suggest a name for a data science podcast. Answer with the name only.", run three calls at each of `temperature` 0, 0.5 and 1.0.
Build a DataFrame and count the number of *distinct* answers per temperature.

</div>

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

### Exercise 1.5 [G9] Classify 20 support tickets (45 min)

The list below holds 20 IT help-desk tickets. Write a JSON schema with two fields, `category` (one of `hardware`, `software`,
`account`, `network`, `other`) and `urgency` (`low`, `medium`, `high`), include short category definitions in the prompt,
tag every ticket with structured output, build a DataFrame, and plot the number of tickets per category. Cache the results to a CSV in `DATA_DIR` so the cell is cheap to re-run.

```python
tickets = [
    "My laptop will not turn on after the update",
    "I cannot log in to the university email",
    "Excel crashes every time I open a file larger than 50 MB",
    "The Wi-Fi in the library drops every ten minutes",
    "Please reset my password, I forgot it",
    "The projector in room B12 shows no signal",
    "Teams says my account is disabled",
    "Python is not recognised as a command in the terminal",
    "The printer on the third floor is out of toner",
    "VPN connects but I cannot reach the file server",
    "My screen flickers when I plug in the external monitor",
    "I need access to the shared research folder",
    "Jupyter opens but the kernel keeps dying",
    "The classroom PCs boot to a black screen",
    "Two-factor codes never arrive on my phone",
    "Zoom audio is choppy on the campus network",
    "My keyboard types the wrong characters",
    "Git push asks for a password and then fails",
    "The e-class site says my session expired",
    "Battery drains in one hour even when idle",
]
```

</div>

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

### Exercise 1.6 [G10] A Greek tutor chain and its rendered prompt (15 min)

Build a LangChain chain whose system prompt makes the model answer in Greek in at most two sentences. Print the rendered system and user messages the model will receive (as in `lec_01d` section 2),
then run the chain on a course question.

</div>

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

### Exercise 1.7 [O2] Hosted inference swap (45 min, stretch)

Following `lec_01f`, configure a hosted OpenAI-compatible endpoint in `.env` (Hugging Face inference providers with your `HF_TOKEN`,
or any other provider), run the chain from Exercise 1.6 against it, and compare: latency, tokens billed, and whether the Greek is better.
Write three sentences on which requirements changed.

</div>

In [ ]:
# Your code here

<div style="text-align: justify; max-width: 90ch">

### Exercise 1.8 [O1] The model without the server (90 min, stretch)

Following `lec_01e`, load `Qwen/Qwen3-0.6B` with `transformers`, generate 60 tokens for the same prompt with and without Ollama,
and report memory (parameters × bytes), tokens per second, and whether the two answers agree.

</div>

In [ ]:
# Your code here